# Lesson 03 — Building makemore Part 2: MLP

- **GitHub issue:** [#3](https://github.com/majorgilles/karpathy_ml_course/issues/3)
- **Video:** https://youtu.be/TCH_1BHY58I
- **Lesson guide:** [../README.md](../README.md)
- **Transcript:** [../transcript.md](../transcript.md)

Use this notebook for exploratory follow-along work. Move reusable code to `../src/`, lightweight checks to `../tests/`, and representative outputs to `../artifacts/`.

## 1. Load names and prepare the MLP workspace

This lesson moves beyond a bigram model: the MLP will use a fixed number of previous characters, called a **context window**, to predict the next character. First load the names as Python strings and establish the small set of libraries used for tensors, one-hot encoding, and later visualizations.

Each element of `words` is one name. The early cells inspect a few names and the dataset size before converting characters into numeric model inputs.


In [1]:
import torch  # Tensor operations and model parameters.
import torch.nn.functional as F  # One-hot encoding and other neural-network helpers.
import matplotlib.pyplot as plt  # Visualizations used later in the lesson.
from sympy.codegen.ast import float32  # Current exploration import; not used by these cells yet.
%matplotlib inline

In [2]:
# Load every name; each line in names.txt becomes one training sequence.
from pathlib import Path

candidate_paths = [
    Path("data/raw/names.txt"),  # Kernel launched from the repository root.
    Path("../../../data/raw/names.txt"),  # Kernel launched from this notebook folder.
]
names_path = next(path for path in candidate_paths if path.exists())
words = names_path.read_text(encoding="utf-8").splitlines()

words[:8]  # Inspect a small sample before building numeric examples.

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
# Confirm how many complete name sequences are available.
len(words)

32033

## 2. Map characters to model-friendly IDs

Neural networks work with numbers, not Python characters. `stoi` means string-to-index and maps each character to one stable integer; `itos` reverses that lookup for readable examples and generated output.

The boundary token `.` receives index `0`. It represents both left padding before a name and the end of a name, so the context window can start before any real letters have appeared.


In [4]:
# Collect the 26 lowercase letters once and sort them for reproducible IDs.
chars = sorted(list(set(''.join(words))))
# Reserve 0 for the boundary token, so letters begin at index 1.
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
# Invert the mapping: model indices back to printable characters.
itos = {i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


## 3. Turn names into fixed-context training examples

`block_size = 3` means every input row contains exactly three previous character IDs. `X` stores those three-character contexts and `Y` stores the one next-character target that followed each context.

For `emma`, the first example is `... → e`, then `..e → m`, and so on until `mma → .`. The initial three zeros are left padding with the boundary token. `words[:5]` deliberately keeps this printed walkthrough small; use all of `words` when building the full training dataset.


In [5]:
# Build (context, next-character) examples from five names for an inspectable walkthrough.
block_size = 3  # Number of preceding characters the model receives as input.
X, Y = [], []  # X holds context rows; Y holds one next-character target per row.

for w in words[:5]:  # Use all `words` later for the full training dataset.
    print(w)
    context = [0] * block_size  # Start with three boundary-token IDs: "...".

    for ch in w + ".":  # Include the final boundary token as a target.
        ix = stoi[ch]  # Integer ID of the character this context should predict.
        X.append(context)
        Y.append(ix)
        print("".join(itos[i] for i in context), "--->", itos[ix])
        context = context[1:] + [ix]  # Drop the oldest ID and append this target ID.

# Convert Python lists to integer tensors for embedding lookup in the MLP.
X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [6]:
# Verify: one three-ID context per example and one integer next-character target.
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

## 4. Give each character a learnable embedding

An **embedding** is a short vector that represents a category. `C` has one row for each of the 27 tokens and two columns for this small visual example, so `C[index]` retrieves that character's current two-number representation.

The values begin randomly and will later be learned with the rest of the MLP. The one-hot cells below verify that direct indexing `C[5]` and matrix multiplication `one_hot @ C` return exactly the same embedding vector.


In [7]:
# Create a 27-row embedding table; each token initially receives two random features.
C = torch.randn((27, 2))

In [12]:
# Encode token index 5 (the character `e`) as a 27-position floating-point one-hot selector.
one_hot = F.one_hot(torch.tensor(5), num_classes=27).float()
# Confirm there is one selector value for each vocabulary token.
one_hot.shape

torch.Size([27])

In [9]:
# Direct embedding lookup: retrieve the two features stored for token index 5.
C[5]

tensor([ 1.4433, -1.3434])

In [13]:
# Matrix-multiply the one-hot selector by C to select row 5 and return its two embedding features.
one_hot @ C

tensor([ 1.4433, -1.3434])

### One-hot matrix multiplication is an embedding lookup

The shapes are `(27,) @ (27, 2) → (2,)`: the one-hot vector has one `1` at index 5 and `0` elsewhere, so the matrix multiplication keeps only row 5 of `C`. That is why `one_hot @ C` produces the same two numbers as `C[5]`.

Direct indexing is the clearer and more efficient form, so later code will use `C[X]` to retrieve embeddings for every token ID in every context row at once.
